# 🎵 Drum & Bass Copy — Google Colab 実行ノートブック

楽曲ファイル（wav / mp3）からドラム・ベースパートを分離し、MIDIファイルとして出力します。

```
入力音声 → [Demucs: stem分離] → ドラム/ベース wav → [MT3: 自動採譜] → MIDI
```

## 実行前の準備

1. **GPUランタイムを有効化**  
   「ランタイム」→「ランタイムのタイプを変更」→「ハードウェアアクセラレータ: GPU（T4）」

2. **GitHub から自動取得**  
   セル 4 を実行すると `https://github.com/Hukumaro/drum-copy` から自動的にクローン / pull されます

3. **入力音声を配置**  
   `マイドライブ/drum-copy_data/input/` に処理したい wav / mp3 ファイルを入れる

4. **上から順にセルを実行**（「ランタイム」→「すべてのセルを実行」でも可）

> 出力 MIDI は `マイドライブ/drum-copy_data/output/` に保存されます。  
> ファイル名は `{曲名}_drums.mid` / `{曲名}_bass.mid` の形式です。

## 1. GPU 確認

In [1]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("⚠️  GPU が検出されません。")
    print("「ランタイム」→「ランタイムのタイプを変更」→「ハードウェアアクセラレータ: GPU」を設定してください。")
    print("CPU でも実行できますが、処理に数十分かかる場合があります。")

✅ GPU: Tesla T4  (15.6 GB VRAM)


## 2. パッケージインストール

> ランタイムが変わるたびに再実行が必要です。初回は数分かかります。

In [ ]:
[
    "import subprocess, sys, os\n",
    "from pathlib import Path\n",
    "\n",
    "def _run(*args, **kwargs):\n",
    "    result = subprocess.run(\n",
    "        [sys.executable, \"-m\", \"pip\", \"install\", \"-q\", *args],\n",
    "        capture_output=True, text=True, **kwargs\n",
    "    )\n",
    "    if result.returncode != 0:\n",
    "        print(result.stderr[-3000:] or result.stdout[-3000:])\n",
    "        raise subprocess.CalledProcessError(result.returncode, result.args)\n",
    "\n",
    "# ── YourMT3+ (PyTorch-based AMT model) ──────────────────────────────────────\n",
    "YMT3_DIR = Path(os.environ.get(\"YMT3_DIR\", \"/tmp/ymt3\"))\n",
    "\n",
    "if not (YMT3_DIR / \"amt\" / \"src\").exists():\n",
    "    print(\"Cloning YourMT3+ ...\")\n",
    "    YMT3_DIR.parent.mkdir(parents=True, exist_ok=True)\n",
    "    r = subprocess.run(\n",
    "        [\"git\", \"clone\", \"--depth=1\",\n",
    "         \"https://huggingface.co/spaces/mimbres/YourMT3\",\n",
    "         str(YMT3_DIR)],\n",
    "        capture_output=True, text=True,\n",
    "    )\n",
    "    if r.returncode != 0:\n",
    "        raise RuntimeError(\"YourMT3+ clone failed: \" + r.stderr)\n",
    "    print(\"Installing YourMT3+ requirements...\")\n",
    "    _run(\"-r\", str(YMT3_DIR / \"requirements.txt\"))\n",
    "    print(f\"  YourMT3+ ready: {YMT3_DIR}\")\n",
    "else:\n",
    "    print(f\"  YourMT3+ already present: {YMT3_DIR}\")\n",
    "\n",
    "# ── Stem separation & audio I/O ──────────────────────────────────────────────\n",
    "print(\"Installing demucs, soundfile...\")\n",
    "_run(\"demucs\", \"soundfile\")\n",
    "\n",
    "# ── Bass transcription ─────────────────────────────────────────────────────\n",
    "print(\"Installing basic-pitch (ONNX backend)...\")\n",
    "# basic-pitch declares tensorflow<2.15.1 in its metadata, which is unavailable on\n",
    "# Python 3.11+. --no-deps bypasses this; basic-pitch auto-falls-back to ONNX at\n",
    "# runtime when TF is absent. onnxruntime and pretty_midi are the only runtime deps\n",
    "# not already covered by the YourMT3+ requirements.\n",
    "_run(\"--prefer-binary\", \"onnxruntime\", \"pretty_midi\")\n",
    "_run(\"--prefer-binary\", \"--no-deps\", \"basic-pitch\")\n",
    "\n",
    "print(\"✅ Installation complete\")\n"
]


## 3. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive マウント完了")

## 4. プロジェクト読み込み（GitHub から常に最新を取得）

GitHub リポジトリ `Hukumaro/drum-copy` から自動的に `git pull` または `git clone` を行い、
Google Drive 内のコードを常に最新状態に保ちます。  
`input/` / `output/` フォルダのユーザーファイルは保護されます。

In [ ]:
import sys, subprocess, shutil
from pathlib import Path

# ── Google Drive 上のプロジェクトルート ─────────────────────────────
# 配置場所を変えた場合はここを修正してください
PROJECT_ROOT = Path("/content/drive/MyDrive/drum-copy")
GITHUB_REPO  = "https://github.com/Hukumaro/drum-copy"
# ────────────────────────────────────────────────────────────────────

def _run(cmd, cwd=None):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result.stdout.strip()

def _backup_user_dirs():
    """input/ output/ をコンテンツ一時領域に退避する"""
    saved = {}
    for d in ["input", "output"]:
        src = PROJECT_ROOT / d
        if src.exists():
            tmp = Path(f"/content/_bak_{d}")
            if tmp.exists():
                shutil.rmtree(str(tmp))
            shutil.copytree(str(src), str(tmp))
            saved[d] = tmp
    return saved

def _restore_user_dirs(saved):
    for d, tmp in saved.items():
        dest = PROJECT_ROOT / d
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(tmp), str(dest), dirs_exist_ok=True)
        shutil.rmtree(str(tmp))

if (PROJECT_ROOT / ".git").exists():
    print(f"🔄 git fetch + reset — {GITHUB_REPO}")
    _run(["git", "fetch", "origin"], cwd=str(PROJECT_ROOT))
    out = _run(["git", "reset", "--hard", "origin/main"], cwd=str(PROJECT_ROOT))
    print(out)
    print("✅ 最新コードに更新しました")
elif PROJECT_ROOT.exists():
    print("⚠️  .git が見つかりません。input/output を退避して再クローンします...")
    saved = _backup_user_dirs()
    shutil.rmtree(str(PROJECT_ROOT))
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(_run(["git", "clone", GITHUB_REPO, str(PROJECT_ROOT)]))
    _restore_user_dirs(saved)
    print("✅ クローン完了（input/output は保持）")
else:
    print(f"📥 初回クローン — {GITHUB_REPO}")
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(_run(["git", "clone", GITHUB_REPO, str(PROJECT_ROOT)]))
    print("✅ クローン完了")

# git reset 後、古い .pyc が再利用されないよう __pycache__ を削除する
for _cache in PROJECT_ROOT.rglob("__pycache__"):
    shutil.rmtree(_cache, ignore_errors=True)

sys.path.insert(0, str(PROJECT_ROOT))
%cd {PROJECT_ROOT}

# モジュールキャッシュを破棄して最新コードを確実に再インポートする
for _mod in list(sys.modules.keys()):
    if _mod.startswith(("pipeline", "backends")):
        del sys.modules[_mod]

print(f"\n✅ プロジェクトパス: {PROJECT_ROOT}")

## 5. パス設定

入力・出力ディレクトリは **コードとは別フォルダ** `drum-copy_data/` に保存されます。  
**通常はここを変更する必要はありません。**

| フォルダ | Google Drive 上のパス |
|---------|---------------------|
| 入力音声 | `マイドライブ/drum-copy_data/input/` |
| 出力 MIDI | `マイドライブ/drum-copy_data/output/` |
| コード | `マイドライブ/drum-copy/` |

In [ ]:
from backends.colab import ColabBackend

# ── 設定 ────────────────────────────────────────────────────────────
MODEL        = "htdemucs"         # Demucs モデル（htdemucs / htdemucs_ft / mdx_extra など）
TARGET_STEMS = ["drums", "bass"]  # 抽出・採譜する stem
OVERWRITE    = False              # True にすると既存の MIDI を上書き
# ────────────────────────────────────────────────────────────────────

# コードとは別フォルダにユーザーデータを保持する（案 B）
DATA_ROOT = Path("/content/drive/MyDrive/drum-copy_data")

backend = ColabBackend(
    data_root=str(DATA_ROOT),
    use_gdrive=False,  # Drive は既にマウント済み
    tmp_dir="/content/tmp/drum-copy",
)
backend.setup()
paths = backend.get_paths()

SUPPORTED = {".wav", ".mp3"}
audio_files = [p for p in sorted(paths.input_dir.iterdir()) if p.suffix.lower() in SUPPORTED]

print(f"入力ディレクトリ : {paths.input_dir}")
print(f"出力ディレクトリ : {paths.output_dir}")
print(f"一時ディレクトリ : {paths.tmp_dir}")
print(f"モデル          : {MODEL}")
print(f"対象 stem       : {TARGET_STEMS}")
print()
print(f"入力ファイル ({len(audio_files)} 件):")
for f in audio_files:
    print(f"  - {f.name}")
if not audio_files:
    print(f"⚠️  {paths.input_dir} に音声ファイルが見つかりません。")

### (オプション) Google Drive を使わず直接アップロードする場合

下のセルのコメントを外して実行すると、PCからファイルをアップロードできます。

In [ ]:
# from google.colab import files as colab_files
# import shutil
#
# uploaded = colab_files.upload()
# for fname in uploaded:
#     dest = paths.input_dir / fname
#     shutil.move(fname, dest)
#     print(f"保存: {dest}")
#
# # アップロード後に audio_files を更新
# audio_files = [p for p in sorted(paths.input_dir.iterdir()) if p.suffix.lower() in SUPPORTED]

## 6. パイプライン実行

- **Step 1** : Demucs で指定 stem（ドラム・ベースなど）を一括分離 → `tmp/` に保存  
- **Step 2** : MT3 で各 stem を採譜 → `output/` に MIDI 保存（例: `song_drums.mid`, `song_bass.mid`）

> 1曲あたりの目安: GPU(T4) で stem 1本につき 2〜5分、CPU では 20〜60分

In [ ]:
# YourMT3+ is PyTorch-based — no TensorFlow ABI workarounds needed.
print("✅ Environment ready (no TF ABI stubs required)")


In [ ]:
import logging
import shutil

from pipeline.stem_separation.separator import separate
from pipeline.transcription.transcriber import transcribe as _transcribe_drums
from pipeline.transcription.bass_transcriber import transcribe_bass as _transcribe_bass

_TRANSCRIBERS = {"drums": _transcribe_drums, "bass": _transcribe_bass}


def _dispatch(wav_path, output_dir, stem_name):
    fn = _TRANSCRIBERS.get(stem_name)
    if fn is None:
        raise ValueError(f"No transcriber for stem: {stem_name!r}")
    return fn(wav_path, output_dir, stem_name=stem_name)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("drum_copy")

if not audio_files:
    raise FileNotFoundError(f"入力ファイルがありません: {paths.input_dir}")

results = []
for audio_path in audio_files:
    # 全 stem の出力パスを確認して全てスキップ判定
    midi_outs = {
        stem: paths.output_dir / f"{audio_path.stem}_{stem}.mid"
        for stem in TARGET_STEMS
    }
    pending_stems = [
        stem for stem, midi_out in midi_outs.items()
        if not midi_out.exists() or OVERWRITE
    ]

    if not pending_stems:
        logger.info("スキップ（全 stem 既存）: %s", audio_path.name)
        for stem, midi_out in midi_outs.items():
            results.append((audio_path.name, stem, str(midi_out), "SKIP"))
        continue

    try:
        logger.info("=== Step 1: Stem 分離 — %s  stems=%s ===", audio_path.name, pending_stems)
        stem_wavs = separate(audio_path, paths.tmp_dir, model=MODEL, stems=pending_stems)

        for stem_name, wav_path in stem_wavs.items():
            midi_out = midi_outs[stem_name]
            try:
                logger.info("=== Step 2: MIDI 変換 — %s [%s] ===", audio_path.name, stem_name)
                midi_path = _dispatch(wav_path, paths.output_dir, stem_name=stem_name)

                # transcriber が wav stem 名で保存するので、曲名ベースにリネーム
                expected = paths.output_dir / f"{wav_path.stem}_{stem_name}.mid"
                if midi_path != midi_out and midi_path.exists():
                    midi_path.rename(midi_out)
                    midi_path = midi_out

                results.append((audio_path.name, stem_name, str(midi_path), "OK"))
                logger.info("完了: %s", midi_path)

            except Exception as exc:
                logger.error("失敗 %s [%s]: %s", audio_path.name, stem_name, exc, exc_info=True)
                results.append((audio_path.name, stem_name, "", f"ERROR: {exc}"))

    except Exception as exc:
        # exc_info=True でフルトレースバックを出力（デバッグ用）
        logger.error("Stem 分離失敗 %s: %s", audio_path.name, exc, exc_info=True)
        for stem in pending_stems:
            results.append((audio_path.name, stem, "", f"ERROR: {exc}"))
    finally:
        stem_tmp = paths.tmp_dir / audio_path.stem
        if stem_tmp.exists():
            shutil.rmtree(stem_tmp)

print("\n─── 結果 ─────────────────────────────────────")
for src, stem, dst, status in results:
    icon = "✅" if status in ("OK", "SKIP") else "❌"
    label = f"{src}  [{stem}]"
    print(f"{icon}  {label}  →  {dst or status}")


## 7. MIDI ダウンロード（オプション）

出力 MIDI は Google Drive の `output/` に自動保存されています。  
Colab から直接ダウンロードしたい場合は下のセルを実行してください。

In [ ]:
from google.colab import files as colab_files

midi_files = list(paths.output_dir.glob("*.mid"))
if midi_files:
    for f in midi_files:
        print(f"ダウンロード中: {f.name}")
        colab_files.download(str(f))
else:
    print("出力 MIDI ファイルが見つかりません。")